In [1]:
# ============================================================================
# 03_attribution_evidence.ipynb
# Sample analysis cases, compute SHAP and LIME attributions under every
# (model x attribution-method) combination, and export the structured evidence
# blocks handed to the LLMs in 06_generation. Cases are stratified by predicted
# risk with a deliberately over-represented near-threshold stratum. All LIME
# settings are stated explicitly (perturbation sample size, kernel width,
# discretisation) so the attribution stage is fully specified.
# ============================================================================

# [Imports and global configuration]
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
from lime.lime_tabular import LimeTabularExplainer
from joblib import Parallel, delayed

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

ROOT = Path("..")
DATA_DIR = ROOT / "data"
PROC_DIR = DATA_DIR / "processed"
MODEL_DIR = DATA_DIR / "models"
FIG_DIR = ROOT / "results" / "figures"
TAB_DIR = ROOT / "results" / "tables"
for d in (FIG_DIR, TAB_DIR):
    d.mkdir(parents=True, exist_ok=True)

# [Plotting style: grayscale, publication settings, no captions]
sns.set_theme(style="whitegrid", context="paper")
sns.set_palette(sns.color_palette("gray", n_colors=8))
plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "font.family": "sans-serif",
    "axes.grid": True, "grid.color": "0.85", "grid.linewidth": 0.5,
    "axes.edgecolor": "0.3", "axes.labelcolor": "0.15", "text.color": "0.15",
    "xtick.color": "0.3", "ytick.color": "0.3", "image.cmap": "gray",
    "savefig.bbox": "tight", "savefig.transparent": False,
})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)
    print(f"saved figure: {name}")

def save_table(df, name, index=False):
    df.to_csv(TAB_DIR / f"{name}.csv", index=index, encoding="utf-8-sig")
    print(f"saved table: {name}")

# [Study configuration]
with open(DATA_DIR / "feature_registry.json", encoding="utf-8") as f:
    registry = json.load(f)
DOMAINS = list(registry.keys())
MODELS = {"XGBoost": "xgboost", "LightGBM": "lightgbm", "RandomForest": "random_forest"}
XAI_METHODS = ["SHAP", "LIME"]

# Paper-scale run uses N_CASES_MAIN = 300 and LIME_N_SAMPLES = 2000; the values
# below keep the attribution stage tractable on a modest (2-core) machine.
N_CASES_MAIN = 300         # cases sampled per domain for the main blocks
TOP_K = 5                 # number of drivers handed to the generator
DECISION_THRESHOLD = 0.5
BOUNDARY_WIDTH = 0.05     # |p - threshold| <= width defines the boundary stratum
BOUNDARY_SHARE = 0.34     # fraction of the sample drawn from the boundary stratum
REFERENCE_MODEL = "XGBoost"

# LIME settings (stated explicitly for reproducibility)
LIME_N_SAMPLES = 2000     # perturbation samples per instance (paper uses 2000)
LIME_KERNEL_WIDTH = None  # None -> LIME default sqrt(n_features)*0.75
LIME_DISCRETIZE = True    # discretise continuous features (LIME default)

# Adverse-decision ("rescue target") subgroup, reported ALONGSIDE the full
# sample rather than replacing it. Defined only for the credit domains; the
# control (churn) and income (adult) domains have no adverse-action analog and
# are flagged not-applicable. An explicit adverse stratum guarantees enough
# adverse cases for the subgroup analysis even in low-prevalence domains.
ADVERSE_APPLICABLE = {"credit_personal", "credit_corporate", "credit_taiwan", "credit_gmsc"}
ADVERSE_MIN_SHARE = 0.40

def is_adverse(domain, ref_prob):
    # The subject who would receive the unfavourable decision. Corporate target
    # positive = investment grade, so adverse = predicted speculative (p < 0.5);
    # the other credit targets positive = default/delinquency, so adverse =
    # predicted positive (p >= 0.5).
    if domain not in ADVERSE_APPLICABLE:
        return False
    return ref_prob < DECISION_THRESHOLD if domain == "credit_corporate" else ref_prob >= DECISION_THRESHOLD

def load_arrays(domain):
    feats = registry[domain]["features"]
    tr = pd.read_parquet(PROC_DIR / f"{domain}_train.parquet")
    te = pd.read_parquet(PROC_DIR / f"{domain}_test.parquet")
    pred = pd.read_parquet(PROC_DIR / f"{domain}_test_predictions.parquet")
    return feats, tr, te, pred

def sample_cases(domain, pred):
    # Stratify by the reference model's predicted probability, over-representing
    # (i) the near-threshold boundary stratum and (ii) the adverse-decision
    # stratum, then fill the remainder across deciles. Returns row positions.
    p = pred[MODELS[REFERENCE_MODEL]].values
    rng = np.random.RandomState(SEED)
    n = min(N_CASES_MAIN, len(p))
    all_idx = np.arange(len(p))
    adverse_mask = np.array([is_adverse(domain, pv) for pv in p])
    # (i) Boundary stratum.
    boundary = np.where(np.abs(p - DECISION_THRESHOLD) <= BOUNDARY_WIDTH)[0]
    n_boundary = min(len(boundary), int(round(n * BOUNDARY_SHARE)))
    pick_b = rng.choice(boundary, n_boundary, replace=False) if n_boundary else np.array([], int)
    chosen = pick_b
    # (ii) Adverse stratum top-up (credit domains only).
    if domain in ADVERSE_APPLICABLE:
        adverse_pool = np.setdiff1d(np.where(adverse_mask)[0], chosen)
        already = int(adverse_mask[chosen].sum()) if len(chosen) else 0
        n_adv = max(0, min(len(adverse_pool), int(round(n * ADVERSE_MIN_SHARE)) - already))
        pick_a = rng.choice(adverse_pool, n_adv, replace=False) if n_adv else np.array([], int)
        chosen = np.concatenate([chosen, pick_a])
    # (iii) Fill the remainder stratified across deciles of predicted probability.
    rest_pool = np.setdiff1d(all_idx, chosen)
    n_rest = n - len(chosen)
    if n_rest > 0 and len(rest_pool):
        deciles = np.clip((p[rest_pool] * 10).astype(int), 0, 9)
        pick_r = []
        for dstrat in range(10):
            idx = rest_pool[deciles == dstrat]
            if len(idx) == 0:
                continue
            take = max(1, int(round(n_rest / 10)))
            pick_r.append(rng.choice(idx, min(take, len(idx)), replace=False))
        if pick_r:
            chosen = np.concatenate([chosen, np.concatenate(pick_r)])
    return np.sort(np.unique(chosen))[:n]

def norm_contrib(vals):
    # Normalise contributions by the total absolute mass of the retained drivers.
    denom = np.sum(np.abs(vals)) or 1.0
    return [round(float(v / denom), 4) for v in vals]

def lime_one(explainer, model, row, feats, base_value):
    # Explain a single instance with LIME and return a top-k driver list.
    exp = explainer.explain_instance(row, model.predict_proba,
                                     num_features=TOP_K, num_samples=LIME_N_SAMPLES,
                                     labels=(1,))
    pairs = exp.as_map()[1]                      # (feature_index, weight)
    pairs = sorted(pairs, key=lambda t: -abs(t[1]))[:TOP_K]
    vals = [w for _, w in pairs]
    norms = norm_contrib(vals)
    drivers = []
    for rank, ((fi, w), wn) in enumerate(zip(pairs, norms), start=1):
        drivers.append({"feature": feats[fi], "contribution": round(float(w), 6),
                        "contribution_norm": wn,
                        "direction": "increases" if w > 0 else "decreases", "rank": rank})
    return drivers

# [Compute attributions and build evidence blocks]
evidence = []
overlap_rows = []
for domain in DOMAINS:
    feats, tr, te, pred = load_arrays(domain)
    Xtr = tr[feats].values.astype(float)
    Xte = te[feats].values.astype(float)
    sel = sample_cases(domain, pred)
    case_ids = te["__case_id"].values[sel]
    # Case-level adverse flag from the reference model's prediction (consistent
    # across model/XAI variants of the same case).
    ref_prob_sel = pred[MODELS[REFERENCE_MODEL]].values[sel]
    adverse_map = {str(cid): bool(is_adverse(domain, pv))
                   for cid, pv in zip(case_ids, ref_prob_sel)}
    adverse_app = domain in ADVERSE_APPLICABLE
    # One LIME explainer per domain (depends only on the training data).
    lime_expl = LimeTabularExplainer(
        Xtr, feature_names=feats, class_names=["negative", "positive"],
        discretize_continuous=LIME_DISCRETIZE, kernel_width=LIME_KERNEL_WIDTH,
        mode="classification", random_state=SEED)
    for disp, fname in MODELS.items():
        model = joblib.load(MODEL_DIR / f"{domain}_{fname}.joblib")
        base_value = round(float(model.predict_proba(Xtr)[:, 1].mean()), 4)
        proba_sel = model.predict_proba(Xte[sel])[:, 1]
        # SHAP (TreeExplainer, probability scale).
        expl = shap.TreeExplainer(model, feature_perturbation="tree_path_dependent")
        sv = expl.shap_values(Xte[sel])
        sv = sv[1] if isinstance(sv, list) else sv
        if sv.ndim == 3:
            sv = sv[:, :, 1]
        for i, cid in enumerate(case_ids):
            order = np.argsort(-np.abs(sv[i]))[:TOP_K]
            vals = sv[i][order]; norms = norm_contrib(vals)
            drivers = [{"feature": feats[j], "contribution": round(float(v), 6),
                        "contribution_norm": wn,
                        "direction": "increases" if v > 0 else "decreases", "rank": r}
                       for r, (j, v, wn) in enumerate(zip(order, vals, norms), start=1)]
            evidence.append({"case_id": str(cid), "domain": domain, "model": disp,
                             "xai": "SHAP", "base_value": base_value,
                             "predicted_probability": round(float(proba_sel[i]), 4),
                             "adverse": adverse_map[str(cid)], "adverse_applicable": adverse_app,
                             "drivers": drivers})
        # LIME (parallel across the sampled cases).
        lime_drivers = Parallel(n_jobs=2)(
            delayed(lime_one)(lime_expl, model, Xte[sel][i], feats, base_value)
            for i in range(len(sel)))
        for i, cid in enumerate(case_ids):
            evidence.append({"case_id": str(cid), "domain": domain, "model": disp,
                             "xai": "LIME", "base_value": base_value,
                             "predicted_probability": round(float(proba_sel[i]), 4),
                             "adverse": adverse_map[str(cid)], "adverse_applicable": adverse_app,
                             "drivers": lime_drivers[i]})
    print(f"{domain:16s} cases={len(sel)} evidence_blocks so far={len(evidence)}")

# Persist the evidence corpus.
with open(DATA_DIR / "evidence_blocks.json", "w", encoding="utf-8") as f:
    json.dump(evidence, f, ensure_ascii=False)
print(f"saved evidence_blocks.json ({len(evidence):,} blocks)")

# [SHAP vs LIME agreement per domain]
ev = pd.DataFrame(evidence)
def feat_set(drv):
    return frozenset(d["feature"] for d in drv)
def dir_map(drv):
    return {d["feature"]: d["direction"] for d in drv}

agree_rows = []
for domain in DOMAINS:
    for disp in MODELS:
        sub = ev[(ev.domain == domain) & (ev.model == disp)]
        s = sub[sub.xai == "SHAP"].set_index("case_id")["drivers"]
        l = sub[sub.xai == "LIME"].set_index("case_id")["drivers"]
        jac, dira = [], []
        for cid in s.index.intersection(l.index):
            a, b = feat_set(s[cid]), feat_set(l[cid])
            jac.append(len(a & b) / len(a | b) if (a | b) else 0.0)
            da, db = dir_map(s[cid]), dir_map(l[cid])
            shared = set(da) & set(db)
            if shared:
                dira.append(np.mean([da[f] == db[f] for f in shared]))
        agree_rows.append({"domain": domain, "model": disp,
                           "feature_jaccard": round(float(np.mean(jac)), 3),
                           "feature_distance": round(1 - float(np.mean(jac)), 3),
                           "direction_agreement": round(float(np.mean(dira)) if dira else np.nan, 3)})
agree = pd.DataFrame(agree_rows)
save_table(agree, "tab03_shap_lime_agreement")

# [Figure 07: SHAP-LIME feature-set overlap by domain (reference model)]
ref = agree[agree.model == REFERENCE_MODEL].set_index("domain").loc[DOMAINS]
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.bar(range(len(DOMAINS)), ref["feature_jaccard"], color="0.45",
       edgecolor="0.15", width=0.62)
for i, v in enumerate(ref["feature_jaccard"]):
    ax.text(i, v + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(range(len(DOMAINS)))
ax.set_xticklabels([registry[d]["label"] for d in DOMAINS], rotation=28, ha="right", fontsize=8)
ax.set_ylabel("SHAP-LIME top-5 Jaccard overlap"); ax.set_ylim(0, 1)
save_fig(fig, "fig07_shap_lime_overlap")

# [Figure 08: direction agreement between SHAP and LIME on shared drivers]
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.bar(range(len(DOMAINS)), ref["direction_agreement"], color="0.35",
       edgecolor="0.15", width=0.62)
ax.set_xticks(range(len(DOMAINS)))
ax.set_xticklabels([registry[d]["label"] for d in DOMAINS], rotation=28, ha="right", fontsize=8)
ax.set_ylabel("Direction agreement (shared drivers)"); ax.set_ylim(0, 1.05)
save_fig(fig, "fig08_direction_agreement")

# [Figure 09: top-1 driver frequency in the primary domain, reference model + SHAP]
prim = next(d for d in DOMAINS if registry[d]["primary"])
sub = ev[(ev.domain == prim) & (ev.model == REFERENCE_MODEL) & (ev.xai == "SHAP")]
top1 = pd.Series([d[0]["feature"] for d in sub["drivers"]]).value_counts().head(10)
fig, ax = plt.subplots(figsize=(6.6, 4.0))
ax.barh(range(len(top1))[::-1], top1.values, color="0.5", edgecolor="0.15")
ax.set_yticks(range(len(top1))[::-1]); ax.set_yticklabels(top1.index, fontsize=7)
ax.set_xlabel("Times ranked the top driver")
save_fig(fig, "fig09_top_driver_frequency")

# [Adverse-decision subgroup sizes, reported alongside the full sample]
case_level = (ev[["domain", "case_id", "adverse", "adverse_applicable"]]
              .drop_duplicates(subset=["domain", "case_id"]))
case_level.to_parquet(DATA_DIR / "case_adverse_flags.parquet", index=False)
adv_rows = []
for domain in DOMAINS:
    sub = case_level[case_level.domain == domain]
    adv_rows.append({"domain": domain, "adverse_applicable": bool(sub["adverse_applicable"].iloc[0]),
                     "n_cases": int(len(sub)), "n_adverse": int(sub["adverse"].sum()),
                     "adverse_share": round(float(sub["adverse"].mean()), 3)})
adv = pd.DataFrame(adv_rows)
save_table(adv, "tab08_adverse_subgroup_sizes")

# [Figure 10: full sample vs adverse-decision subgroup size by domain]
fig, ax = plt.subplots(figsize=(7.2, 3.6))
x = np.arange(len(DOMAINS)); w = 0.4
ax.bar(x - w / 2, adv["n_cases"], width=w, color="0.7", edgecolor="0.15", label="full sample")
ax.bar(x + w / 2, adv["n_adverse"], width=w, color="0.3", edgecolor="0.15", label="adverse subgroup")
ax.set_xticks(x); ax.set_xticklabels([registry[d]["label"] for d in DOMAINS], rotation=28, ha="right", fontsize=8)
ax.set_ylabel("Cases"); ax.legend(fontsize=7, frameon=False)
save_fig(fig, "fig10_adverse_subgroup_sizes")
print("adverse subgroup sizes:\n", adv.to_string(index=False))

print("\nAttribution complete:",
      f"{len(evidence):,} evidence blocks across {len(DOMAINS)} domains.")
print(agree.to_string(index=False))


credit_personal  cases=297 evidence_blocks so far=1782
credit_corporate cases=277 evidence_blocks so far=3444
credit_taiwan    cases=297 evidence_blocks so far=5226
credit_gmsc      cases=287 evidence_blocks so far=6948
income_adult     cases=300 evidence_blocks so far=8748
churn            cases=292 evidence_blocks so far=10500
saved evidence_blocks.json (10,500 blocks)
saved table: tab03_shap_lime_agreement
saved figure: fig07_shap_lime_overlap
saved figure: fig08_direction_agreement
saved figure: fig09_top_driver_frequency
saved table: tab08_adverse_subgroup_sizes
saved figure: fig10_adverse_subgroup_sizes
adverse subgroup sizes:
           domain  adverse_applicable  n_cases  n_adverse  adverse_share
 credit_personal                True      297        190          0.640
credit_corporate                True      277        187          0.675
   credit_taiwan                True      297        180          0.606
     credit_gmsc                True      287        170          0.59

credit_gmsc      cases=56 evidence_blocks so far=1338


income_adult     cases=60 evidence_blocks so far=1698


churn            cases=60 evidence_blocks so far=2058
saved evidence_blocks.json (2,058 blocks)
saved table: tab03_shap_lime_agreement


saved figure: fig07_shap_lime_overlap
saved figure: fig08_direction_agreement


saved figure: fig09_top_driver_frequency
saved table: tab08_adverse_subgroup_sizes


saved figure: fig10_adverse_subgroup_sizes
adverse subgroup sizes:
           domain  adverse_applicable  n_cases  n_adverse  adverse_share
 credit_personal                True       55         34          0.618
credit_corporate                True       55         34          0.618
   credit_taiwan                True       57         34          0.596
     credit_gmsc                True       56         34          0.607
    income_adult               False       60          0          0.000
           churn               False       60          0          0.000

Attribution complete: 2,058 evidence blocks across 6 domains.
          domain        model  feature_jaccard  feature_distance  direction_agreement
 credit_personal      XGBoost            0.389             0.611                0.956
 credit_personal     LightGBM            0.439             0.561                0.974
 credit_personal RandomForest            0.455             0.545                0.965
credit_corporate     